<a href="https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions

The action playbook ranks content pages according to the model output.

Each recommendation includes a reason code that explains why the page appears in the queue.

### Reason Codes

- LOW_CTR_HIGH_IMPRESSIONS → Improve CTR
- HIGH_POSITION → Improve Ranking
- LOW_PRIORITY → Monitor

In [14]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded")

Token loaded


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
"""

df = con.sql(query).df()

# Fill missing values
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(0)

# CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Action score
df["action_score"] = (
    df["gsc_impressions"] * (1 - df["ctr"])
)

# Reason codes
df["reason_code"] = np.where(
    (df["gsc_impressions"] > 100) & (df["ctr"] < 0.05),
    "LOW_CTR_HIGH_IMPRESSIONS",
    np.where(
        df["gsc_avg_position"] > 20,
        "HIGH_POSITION",
        "LOW_PRIORITY"
    )
)

# Action labels
mapping = {
    "LOW_CTR_HIGH_IMPRESSIONS": "Improve CTR",
    "HIGH_POSITION": "Improve Ranking",
    "LOW_PRIORITY": "Monitor"
}

df["action"] = df["reason_code"].map(mapping)

queue = df.sort_values(
    "action_score",
    ascending=False
)

queue[
    [
        "content_hash_id",
        "action_score",
        "reason_code",
        "action"
    ]
].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,action_score,reason_code,action
90707,content_36e53e9c707674fc,7165.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
1352,content_fd2117c2c6790e4b,6891.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
8618,content_29c4a3831609805d,5865.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
9080,content_e8b074fd4a082388,4266.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
4102,content_cf651123f1085418,3633.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
1026,content_00d4fdf6e48a2d38,3572.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
91164,content_f7c9fcc26f6e23c1,3455.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
14855,content_62673eea26c31c17,3281.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
36122,content_34a70fea29d15f24,3251.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR
35051,content_b64fb5a4b3f9b7aa,3126.0,LOW_CTR_HIGH_IMPRESSIONS,Improve CTR


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This action playbook helps prioritize content pages for manual review based on search performance signals.

The recommendations are intended for decision support and should be reviewed by a human before taking action.

## Limits

The recommendations are based only on the available dataset.

They do not account for seasonality, recent content updates, content quality, or business priorities.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
playbook = {
    "Intended Use":
        "Prioritize content pages for manual review.",
    "Limits":
        "Decision-support only. Results may not reflect seasonality, recent updates, or business priorities."
}

for key, value in playbook.items():
    print(key)
    print(value)
    print()

Intended Use
Prioritize content pages for manual review.

Limits
Decision-support only. Results may not reflect seasonality, recent updates, or business priorities.



## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review

Before taking action, a reviewer should:

- Check whether the page was recently updated.
- Verify search performance trends.
- Review page quality and search intent.
- Confirm business priorities.

## No-go list

The playbook should not automatically:

- Delete pages.
- Merge content.
- Publish content updates.
- Change titles or metadata without human review.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
human_review = [
    "Check recent content updates",
    "Verify search performance trends",
    "Review page quality",
    "Confirm business priority"
]

no_go = [
    "Do not delete pages automatically",
    "Do not merge content automatically",
    "Do not publish content automatically",
    "Do not change metadata without human review"
]

print("Human Review")
for item in human_review:
    print("-", item)

print("\nNo-Go List")
for item in no_go:
    print("-", item)

Human Review
- Check recent content updates
- Verify search performance trends
- Review page quality
- Confirm business priority

No-Go List
- Do not delete pages automatically
- Do not merge content automatically
- Do not publish content automatically
- Do not change metadata without human review


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring

The recommendations should be reviewed periodically.

Model retraining may be needed if search behavior changes or model performance declines.

## Retrain triggers

- Large changes in CTR or impressions.
- New content is added.
- Search ranking patterns change.
- Model accuracy decreases over time.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring = {
    "Retrain Triggers": [
        "Large changes in CTR",
        "Large changes in impressions",
        "New content added",
        "Search ranking patterns change",
        "Model performance decreases"
    ]
}

for trigger in monitoring["Retrain Triggers"]:
    print("-", trigger)

- Large changes in CTR
- Large changes in impressions
- New content added
- Search ranking patterns change
- Model performance decreases


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports

The ranked action queue is exported to the work/outputs directory.

This file will be reused in the final capstone research paper.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/action_playbook_queue.csv",
    index=False
)

print("Export completed.")
print("Saved to: work/outputs/action_playbook_queue.csv")

Export completed.
Saved to: work/outputs/action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.